# Beschreibung: 

# Importe:

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

# Funktionen:

In [2]:
def discretize(df:pd.DataFrame, bins:int=4, cutoffs:dict=None):
    """
    Numerische Werte werden in diskrete Kategorien überführt:
        - Binarität in Form von Zahlen sind schon Diskret (0/1 → keine Diskretisierung)
        - optionale user-defined Cutoffs (z.B. medizinisch)
        - geringe Range → equal-width diskretisierung
        - ansonsten fallback: qcut
    
    Parameter:
        df: pd.DataFrame = Zu diskreditierende Pandas DataFrame
        bins: int = Anzahl der Diskretisierungsintervalle - Quantils-binning (für fallback)
        cutoffs: dict = dictornary mit {col: [cut1, cut2, ...]} für feste Intervalle
    
    Rückgabe:
        df: DataFrame mit *_disc Spalten
    """
    df = df.copy()
    cutoffs = cutoffs or {}  # wenn None übergeben wird
    
    for col in df.columns:
        if df[col].dtype not in ["float64", "int64"]:
            continue
        
        # 1. binäre Spalte → nicht diskretisieren
        if df[col].nunique() == 2:
            df[col + "_disc"] = df[col]
            continue
        
        # 2. user-defined Cutoffs
        if col in cutoffs:
            bins_list = [-np.inf] + cutoffs[col] + [np.inf]
            
            try:
                df[col + "_disc"] = pd.cut(df[col], bins=bins_list, labels=False)
            except Exception as e:
                print(f"Warnung (Cutoffs für {col} konnten nicht angewandt werden):", e)
                df[col + "_disc"] = pd.cut(df[col], bins=3, labels=False)
            
            continue
        
        # 3. kleiner Wertebereich (z.B. Age 18–24)
        if df[col].max() - df[col].min() < 10 and df[col].nunique() > 3:
            df[col + "_disc"] = pd.cut(df[col], bins=3, labels=False)
            continue
        
        # 4. fallback auf quantile
        if df[col].nunique() <= bins:
            df[col + "_disc"] = df[col].rank(method="dense") - 1
        else:
            df[col + "_disc"] = pd.qcut(
                df[col].rank(method="first"),
                q=bins,
                labels=False,
                duplicates="drop"
            )
    
    return df

In [3]:
def indiscernibility(df, attrs):
    """
    Bildet Äquivalenzklassen der Indiscernibility Relation 'IND(P)' wieder.
    Input:
        • df: ein DataFrame (diskretisiert)
        • attrs: Liste von Attributen, nach denen Objekte verglichen werden
    Output:
        • list: Eine Liste von Listen, wo jede innere Liste eine Äquivalenzklasse [x]P ist.

    Beispiel:
        Input:
            • attrs = ["Age_disc", "StudyTime_disc"]
            df: ...
            Objekt 1: (2, 0)
            Objekt 2: (2, 0)
            Objekt 3: (1, 3)
                ...
        Output:

        [
          [0, 4, 10],     # diese drei Objekte sind ununterscheidbar
          [1, 2],         # diese beiden ebenfalls - ist auch im beispielhaften Input zu sehen.
          [3],            # einzelnes Objekt
          ...
        ]
    """
    groups = {}
    for i, row in df[attrs].iterrows():
        key = tuple(row.tolist())
        groups.setdefault(key, []).append(i)
    return list(groups.values())

In [4]:
def dependency(df, attrs, decision):
    """
    • Berechnet POS_C(D)
    • Zählt, wie viele Objekte eindeutig klassifiziert werden können
    • Dividiert durch Anzahl aller Objekte
    
    Input:
        • df: Information System (diskret)
        • attrs: Konditionsattribute C
        • decision: Entscheidungsattribut D
    Output:
        • Ein float zwischen 0 und 1

    Für die Attribute C:
        • indiscernibility(C) liefert Äquivalenzklassen
            -> Jede Klasse wird geprüft:
                wenn alle Objekte dieselbe Entscheidung haben → Klasse gehört zur positiven Region


    Ein float zwischen 0 und 1:
        • γ = 1 → perfekte Klassifikation
        • γ = 0 → keine Klassifikation möglich
        • 0 < γ < 1 → teils eindeutig, teils unsicher
    """
    if not attrs:
        return 0
    pos = 0
    U = len(df)
    for block in indiscernibility(df, attrs):
        if df.loc[block, decision].nunique() == 1:
            pos += len(block)
    return pos / U

In [5]:
def quick_reduct(df, attrs, decision):
    """
    Was wird gemacht:
        • greedy Auswahl von Attributen
        • Maximiert schrittweise den Dependency Degree γ
        • Endet:
            • Kein Attribut γ verbessern kann. Oder
            • γ voll ist.
    Ursprung: Pawlak (1982) – Grunddefinition Reduct und Shenoi & Yao (1994), Jensen (1998) – QuickReduct-Algorithmus
    
    Input:
        • df: Diskretes IS
        • attrs: Alle konditionalen Attribute
        • decision: Die Zielvariable
    Output:
        • list: Eine Liste von Attributen z.B ['Age_disc'], welches das (hoffentlich:)) minimale notwendige Attributset ist.
    """
    R = []
    gamma_star = dependency(df, attrs, decision)
    gamma_R = 0

    while gamma_R < gamma_star - 1e-12:
        best_attr = None
        best_gamma = gamma_R

        for a in attrs:
            if a in R: continue
            g = dependency(df, R+[a], decision)
            if g > best_gamma + 1e-12:
                best_gamma = g
                best_attr = a

        if best_attr is None:
            break

        R.append(best_attr)
        gamma_R = best_gamma

    return R

In [6]:
def induce_rules(df, reduct, decision):
    """
    Aus jeder Äquivalenzklasse [x]_R:
        Wenn alle Objekte dieselbe Entscheidung haben → Regel: IF (Attribut1 = Wert1 ∧ …) THEN (Decision = Klasse)
    Regeln entstehen genau aus den reinen Blöcken der Reduct-Indiscernibility. (Das entspricht exakt Pawlaks Methode.)
    
    Input:
        • df: Diskretes IS
        • reduct: Vom QuickReduct ausgewählte Atributenset
        • decision: Entscheidungsattribut
    Output:
        • Eine Liste von Regeln

    Beispielhafte Ausgabe:
        {
          'premise': {'Total_Score_disc': 3},
          'decision': 'A',
          'support': 1250
        }
        Dabei gilt es folgendermaßen zu lesen: "IF (premise) THEN (Decision)" und der support gibt die Anzahl der Objekte in dieser Klasse
    """
    rules = []
    for block in indiscernibility(df, reduct):
        decs = df.loc[block, decision].unique()
        if len(decs)==1:
            rules.append({
                "premise": {a: df.loc[block[0], a] for a in reduct},
                "decision": decs[0],
                "support": len(block),
            })
    return rules


# Daten laden:

In [25]:
biased = pd.read_csv("./Kaggle_Daten/Student_Performance_Behavior_Dataset/Students_Grading_Dataset_Biased.csv")
unbiased = pd.read_csv("./Kaggle_Daten/Student_Performance_Behavior_Dataset/Students_Performance_Dataset.csv")

biased["Pass"] = (biased["Grade"].isin(["A","B"])).astype(int)
unbiased["Pass"] = (unbiased["Grade"].isin(["A","B"])).astype(int)


print(biased.shape)
print(unbiased.shape)

(5000, 24)
(5000, 24)


# Ausführung

### Vorbereitung: (Datenaufbereitung)

In [8]:
# Als Entscheidung wird folgendes genutzt:
decision_attr = "Grade"

In [9]:
# Numerische Spalten finden, um diese diskreditieren zu können:
num_cols = biased.select_dtypes(include=["int64","float64"]).columns

In [19]:
# Diskretisierung der numerischen Daten:
biased_disc = discretize_old(biased, num_cols, bins=4)
unbiased_disc = discretize_old(unbiased, num_cols, bins=4)

In [26]:
cutoffs = {
    "age": [20, 22], # Aufteilung in 18/19, 20/21, 22/23/24
    "Sleep_Hours_per_Night": [6, 8], # Aufteilung in 4/5, 6/7, 8/9  -  wenig, Durchschnitt, viel
}

# Diskretisierung der numerischen Daten:
biased_disc = discretize(biased.drop(columns=["Pass"]), bins=4, cutoffs=cutoffs)
unbiased_disc = discretize(unbiased.drop(columns=["Pass"]), bins=4,cutoffs=cutoffs)

# „Pass“ wieder anfügen (NICHT diskretisieren!)
biased_disc["Pass"] = biased["Pass"]
unbiased_disc["Pass"] = unbiased["Pass"]

In [27]:
# Konditionsattribute (Nur numerische Werte: conditional attributs only numeric) 
cond_attrs_o_num = [
    col for col in biased_disc.columns 
    if col.endswith("_disc") and col != decision_attr
]
print(cond_attrs_o_num)

['Age_disc', 'Attendance (%)_disc', 'Midterm_Score_disc', 'Final_Score_disc', 'Assignments_Avg_disc', 'Quizzes_Avg_disc', 'Participation_Score_disc', 'Projects_Score_disc', 'Total_Score_disc', 'Study_Hours_per_Week_disc', 'Stress_Level (1-10)_disc', 'Sleep_Hours_per_Night_disc']


In [28]:
# Konditionsattribute (Alle Werte - bis auf Student_ID und Email: conditional attributs all)
cond_attrs_all = []
for col in biased_disc.columns:
    if col != decision_attr:
        # diskretisierte numerische Werte (z. B. Total_Score_disc)
        if col.endswith("_disc"):
            cond_attrs_all.append(col)
        # direkt kategorische Werte (Gender, Department etc.)
        elif biased_disc[col].dtype == "object":
            cond_attrs_all.append(col)
            
cond_attrs_all.remove('Student_ID') # Das ist ein Identifier, daher muss es raus.
cond_attrs_all.remove('Email') # Es gilt hier dasselbe

print(cond_attrs_all)

['First_Name', 'Last_Name', 'Gender', 'Department', 'Extracurricular_Activities', 'Internet_Access_at_Home', 'Parent_Education_Level', 'Family_Income_Level', 'Age_disc', 'Attendance (%)_disc', 'Midterm_Score_disc', 'Final_Score_disc', 'Assignments_Avg_disc', 'Quizzes_Avg_disc', 'Participation_Score_disc', 'Projects_Score_disc', 'Total_Score_disc', 'Study_Hours_per_Week_disc', 'Stress_Level (1-10)_disc', 'Sleep_Hours_per_Night_disc']


### Datenbetrachtung:

#### cond_attrs_o_num: Datensatz mit nur diskretisierten numerischen Werte

In [29]:
# Redukte
biased_reduct_o_num = quick_reduct(biased_disc, cond_attrs_o_num, decision_attr)
unbiased_reduct_o_num = quick_reduct(unbiased_disc, cond_attrs_o_num, decision_attr)

print("Biased Reduct nur numerische Werte:\n", biased_reduct_o_num)
print("\nUnbiased Reduct nur numerische Werte:\n", unbiased_reduct_o_num)

Biased Reduct nur numerische Werte:
 ['Assignments_Avg_disc', 'Attendance (%)_disc']

Unbiased Reduct nur numerische Werte:
 ['Total_Score_disc']


In [30]:
rules_unbiased_o_num = induce_rules(unbiased_disc, unbiased_reduct_o_num, decision_attr)

for r in rules_unbiased_o_num[:10]:
    print(r)

{'premise': {'Total_Score_disc': np.int64(2)}, 'decision': 'C', 'support': 1250}


In [31]:
biased_pass_reduct_o_num = quick_reduct(biased_disc, cond_attrs_o_num, "Pass")
unbiased_pass_reduct_o_num = quick_reduct(unbiased_disc, cond_attrs_o_num, "Pass")

#biased["Pass"] = (biased[decision_attr].isin(["A","B"])).astype(int)
#unbiased["Pass"] = (unbiased[decision_attr].isin(["A","B"])).astype(int)

rules_pass_biased_o_num = induce_rules(biased_disc, biased_pass_reduct_o_num, "Pass")
rules_pass_unbiased_o_num = induce_rules(unbiased_disc, unbiased_pass_reduct_o_num, "Pass")

#### cond_attrs_all: Datensatz mit nicht nur diskretisierten numerischen Werten sondern auch direkt kategorische Werte (Gender, Department etc.)

In [32]:
# Redukte
biased_reduct_all = quick_reduct(biased_disc, cond_attrs_all, decision_attr)
unbiased_reduct_all = quick_reduct(unbiased_disc, cond_attrs_all, decision_attr)

print("Biased Reduct alle Werte:\n", biased_reduct_all)
print("\nUnbiased Reduct alle Werte:\n", unbiased_reduct_all)

Biased Reduct alle Werte:
 ['Assignments_Avg_disc', 'Attendance (%)_disc']

Unbiased Reduct alle Werte:
 ['Total_Score_disc']


In [33]:
rules_unbiased_all = induce_rules(unbiased_disc, unbiased_reduct_all, decision_attr)

for r in rules_unbiased_all[:10]:
    print(r)

{'premise': {'Total_Score_disc': np.int64(2)}, 'decision': 'C', 'support': 1250}


In [34]:
biased_pass_reduct_all = quick_reduct(biased_disc, cond_attrs_all, "Pass")
unbiased_pass_reduct_all = quick_reduct(unbiased_disc, cond_attrs_all, "Pass")

biased["Pass"] = (biased[decision_attr].isin(["A","B"])).astype(int)
unbiased["Pass"] = (unbiased[decision_attr].isin(["A","B"])).astype(int)

rules_pass_biased_all = induce_rules(biased_disc, biased_pass_reduct_all, "Pass")
rules_pass_unbiased_all = induce_rules(unbiased_disc, unbiased_pass_reduct_all, "Pass")

In [ ]:
#print(biased)
#print(unbiased)

# Resultate

In [35]:
print("Nur numerische Werte:")
print("Biased Reduct:\n", biased_reduct_o_num)
print("\nUnbiased Reduct:\n", unbiased_reduct_o_num)

print("\nPass Reduct (biased):", biased_pass_reduct_o_num)
print("Pass Reduct (unbiased):", unbiased_pass_reduct_o_num)

#print("Pass Rules (biased):", rules_pass_biased_o_num)
#print("Pass Rules (unbiased):", rules_pass_unbiased_o_num)

print("\n")
print("Alle Werte:")
print("Biased Reduct:\n", biased_reduct_all)
print("\nUnbiased Reduct:\n", unbiased_reduct_all)

print("\nPass Reduct (biased):", biased_pass_reduct_all)
print("Pass Reduct (unbiased):", unbiased_pass_reduct_all)

#print("Pass Rules (biased):", rules_pass_biased_all)
#print("Pass Rules (unbiased):", rules_pass_unbiased_all)

Nur numerische Werte:
Biased Reduct:
 ['Assignments_Avg_disc', 'Attendance (%)_disc']

Unbiased Reduct:
 ['Total_Score_disc']

Pass Reduct (biased): ['Attendance (%)_disc', 'Assignments_Avg_disc']
Pass Reduct (unbiased): ['Total_Score_disc']


Alle Werte:
Biased Reduct:
 ['Assignments_Avg_disc', 'Attendance (%)_disc']

Unbiased Reduct:
 ['Total_Score_disc']

Pass Reduct (biased): ['Attendance (%)_disc', 'Assignments_Avg_disc', 'Department', 'First_Name', 'Last_Name', 'Parent_Education_Level', 'Total_Score_disc', 'Midterm_Score_disc', 'Participation_Score_disc']
Pass Reduct (unbiased): ['Total_Score_disc']
